# 02 Bull & bear sentiment picks

Each endpoint is fetched once. The readings compare stored news tone, management outlook, and disclosed fund position changes.

Research data only, not investment advice. Each panel refreshes on a schedule; each panel shows its source date.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

These are member research routes (beta); they are not yet listed in `/api/v1/catalog`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_panel, get_api_key, make_client, nested, selected
client, catalogue = make_client()
api_key = get_api_key()

In [ ]:
score_names = ('ticker', 'picks_pct', 'news_led_pct', 'picks_n_components', 'news_tone_shrunk', 'news_n_articles', 'news_latest_utc', 'mgmt_outlook_direction', 'mgmt_outlook_net_tone', 'mgmt_outlook_filing_date', 'fund_direction', 'fund_as_of', 'agree_count', 'available_count', 'agreement_label')
score_columns = {
    'ticker': 'The market identifier for the company.',
    'picks_pct': 'The stored percentile for the news-required picks composite.',
    'news_led_pct': 'The stored percentile for the news-led component.',
    'picks_n_components': 'The number of components behind the picks reading.',
    'news_tone_shrunk': 'The recency-weighted news tone adjusted toward the market mean.',
    'news_n_articles': 'The number of distinct articles behind the news reading.',
    'news_latest_utc': 'The latest article time in the stored news window.',
    'mgmt_outlook_direction': 'The stored direction classification for management outlook.',
    'mgmt_outlook_net_tone': 'The stored net tone for management outlook.',
    'mgmt_outlook_filing_date': 'The filing date behind the management outlook.',
    'fund_direction': 'The classification of disclosed fund position changes.',
    'fund_as_of': 'The position date behind the disclosed fund changes.',
    'agree_count': 'The number of available signals with the same direction.',
    'available_count': 'The number of signals available for comparison.',
    'agreement_label': 'The plain-language summary of signal agreement.',
}
scores_frame = fetch_panel('News-led sentiment scores', lambda: client.member_sentiment_scores('us', api_key, limit=10), lambda p: [selected(row, score_names) for row in p.get('data') or []], score_columns, source_date=lambda p: nested(p, 'snapshot', '_rebuild_date'), status=lambda p: 'ok; rows returned' if p.get('data') else 'ok; empty', message=lambda p: p.get('note') or (None if p.get('data') else 'No sentiment rows returned.'))

In [ ]:
whale_names = ('side', 'ticker', 'quarter', 'as_of_date', 'n_funds_buying', 'n_funds_selling', 'net_shares_change', 'n_observed_funds', 'n_missing_funds', 'change_basis', 'methodology_version', 'tone', 'n_articles', 'max_published', 'whale_dir', 'tone_label')
whale_columns = {
    'side': 'The bull or bear list assigned by the response.',
    'ticker': 'The market identifier for the company.',
    'quarter': 'The reporting quarter for the disclosed positions.',
    'as_of_date': 'The position date for the disclosed changes.',
    'n_funds_buying': 'The number of observed filers with increased disclosed positions.',
    'n_funds_selling': 'The number of observed filers with reduced disclosed positions.',
    'net_shares_change': 'The disclosed net share change on the stated basis.',
    'n_observed_funds': 'The number of filers observed for this row.',
    'n_missing_funds': 'The number of withheld filers counted for this row.',
    'change_basis': 'The adjustment basis for the disclosed share change.',
    'methodology_version': 'The methodology identifier stored with the row.',
    'tone': 'The stored shrunk news-tone reading.',
    'n_articles': 'The number of distinct articles behind the tone.',
    'max_published': 'The latest article time behind the tone.',
    'whale_dir': 'The API classification attached to the disclosed position change.',
    'tone_label': 'The API classification attached to the news tone.',
}
def whale_rows(payload):
    data = payload.get('data') or {}
    return [{**selected(row, whale_names), 'side': side} for side in ('bull', 'bear') for row in data.get(side) or []]
whale_frame = fetch_panel('Sentiment and disclosed-position shortlists', lambda: client.member_sentiment_whale_picks('us', api_key), whale_rows, whale_columns, source_date=lambda p: nested(p, 'snapshot', 'whale_as_of'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message') or p.get('note'))